In [ ]:
#Part 1-5. train/test split & Dataset

N = X_past.shape[0]
idx = np.random.permutation(N)
train_ratio = 0.8
n_train = int(N*train_ratio)

train_idx = idx[:n_train]
test_idx  = idx[n_train:]

X_past_train   = X_past[train_idx]
X_future_train = X_future[train_idx]
X_past_test    = X_past[test_idx]
X_future_test  = X_future[test_idx]

past_abs_test   = past_abs_arr[test_idx]
future_abs_test = future_abs_arr[test_idx]
base_abs_test   = base_abs_arr[test_idx]

print("Train:", len(train_idx), "Test:", len(test_idx))


class TyphoonDataset(Dataset):
    def __init__(self, Xpast, Xfuture):
        self.Xpast = Xpast.astype(np.float32)
        self.Xfuture = Xfuture.astype(np.float32)
    def __len__(self):
        return len(self.Xpast)
    def __getitem__(self, i):
        return self.Xpast[i], self.Xfuture[i]

BATCH = 64

train_loader = DataLoader(TyphoonDataset(X_past_train, X_future_train),
                          batch_size=BATCH, shuffle=True)
test_loader  = DataLoader(TyphoonDataset(X_past_test,  X_future_test),
                          batch_size=BATCH)


Train: 38700 Test: 9676


In [ ]:
#Part 2. 条件エンコーダ（CNN + BiLSTM）

class CondEncoder(nn.Module):
    def __init__(self, cond_dim=256):
        super().__init__()
        self.conv1 = nn.Conv1d(2, 32, 3, padding=1)
        self.conv2 = nn.Conv1d(32, 32, 3, padding=1)
        self.lstm  = nn.LSTM(32, 64, batch_first=True, bidirectional=True)
        self.fc    = nn.Linear(128, cond_dim)

    def forward(self, x):
        # x : (B, PAST, 2)
        x = x.permute(0,2,1)         # (B,2,PAST)
        x = F.silu(self.conv1(x))
        x = F.silu(self.conv2(x))
        x = x.permute(0,2,1)         # (B,PAST,32)

        _, (h,_) = self.lstm(x)      # h: (2, B, 64)
        h = torch.cat([h[0], h[1]], dim=-1)  # (B,128)
        return F.silu(self.fc(h))          # (B,cond_dim)


Part 2-2. Sinusoidal 時刻埋め込み

In [ ]:
class SinusoidalEmbedding(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.dim = dim
        half = dim // 2
        freq = torch.exp(-torch.arange(half)/half * np.log(10000))
        self.register_buffer("freq", freq)

    def forward(self, t):
        t = t.float().unsqueeze(1)
        ang = t * self.freq.unsqueeze(0)
        return torch.cat([torch.sin(ang), torch.cos(ang)], dim=1)


Part 2-3. ResBlock (1D)

In [ ]:
class ResBlock(nn.Module):
    def __init__(self, in_ch, out_ch, cond_dim):
        super().__init__()
        self.conv_in = nn.Conv1d(in_ch, out_ch, 1)   # 入力を out_ch へ変換
        self.conv1   = nn.Conv1d(out_ch, out_ch, 3, padding=1)
        self.conv2   = nn.Conv1d(out_ch, out_ch, 3, padding=1)
        self.fc_t    = nn.Linear(64, out_ch)
        self.fc_c    = nn.Linear(cond_dim, out_ch)
        self.skip    = nn.Conv1d(in_ch, out_ch, 1)

    def forward(self, x, t_emb, cond):
        x_proj = self.conv_in(x)
        t_add = self.fc_t(t_emb).unsqueeze(-1)
        c_add = self.fc_c(cond).unsqueeze(-1)

        h = x_proj + c_add + t_add
        h = self.conv1(F.silu(h))
        h = self.conv2(F.silu(h))

        return self.skip(x) + h


Part 2-4. UNet1D（未来FUTURE点向け）

In [ ]:
class UNetFuture(nn.Module):
    def __init__(self, cond_dim=256):
        super().__init__()

        self.temb = nn.Sequential(
            SinusoidalEmbedding(64),
            nn.Linear(64,64),
            nn.SiLU(),
            nn.Linear(64,64),
        )

        self.in_conv = nn.Conv1d(2,64,3,padding=1)

        # Down
        self.down = nn.Conv1d(64,128,3,stride=2,padding=1)
        self.res1 = ResBlock(128,128,cond_dim)

        # Up
        self.up   = nn.ConvTranspose1d(128,64,4,stride=2,padding=1)
        self.res2 = ResBlock(128,64,cond_dim)   # ← 修正（128→64）

        self.out_conv = nn.Conv1d(64,2,3,padding=1)

    def forward(self, x, t, cond):
        x = x.permute(0,2,1)
        t_emb = self.temb(t)

        x = self.in_conv(x)
        skip = x.clone()

        x = F.silu(self.down(x))
        x = self.res1(x, t_emb, cond)

        x = F.silu(self.up(x))
        x = torch.cat([x, skip], dim=1)   # (B,128,L)
        x = self.res2(x, t_emb, cond)

        return torch.tanh(self.out_conv(x)).permute(0,2,1)



Part 3. DDPM (T=20) + EMA + 学習

In [ ]:
# --- beta schedule ---
def cosine_beta_schedule(T, s=0.008):
    steps = T+1
    x = np.linspace(0,T,steps)
    alphas_cum = np.cos(((x/T)+s)/(1+s) * np.pi/2)**2
    alphas_cum = alphas_cum / alphas_cum[0]
    betas = 1 - (alphas_cum[1:] / alphas_cum[:-1])
    return np.clip(betas,1e-5,0.999)

class DDPM:
    def __init__(self, T=20):
        self.T = T
        betas = cosine_beta_schedule(T)
        alphas = 1 - betas
        alphas_cum = np.cumprod(alphas)
        alphas_cum_prev = np.concatenate([[1.0], alphas_cum[:-1]])

        self.betas  = torch.tensor(betas, dtype=torch.float32, device=device)
        self.a_bar  = torch.tensor(alphas_cum, dtype=torch.float32, device=device)
        self.a_bar_prev = torch.tensor(alphas_cum_prev, dtype=torch.float32, device=device)

    def q_sample(self, x0, t):
        noise = torch.randn_like(x0)
        a_bar = self.a_bar[t].view(-1,1,1)
        return torch.sqrt(a_bar)*x0 + torch.sqrt(1-a_bar)*noise, noise

ddpm = DDPM(T=20)


# --- EMA ---
def update_ema(model, ema_model, decay=0.999):
    with torch.no_grad():
        for p, ep in zip(model.parameters(), ema_model.parameters()):
            ep.data.mul_(decay).add_(p.data*(1-decay))


# --- Instantiate model ---
cond_encoder = CondEncoder().to(device)
unet = UNetFuture().to(device)
ema_unet = UNetFuture().to(device)
ema_unet.load_state_dict(unet.state_dict())

optimizer = torch.optim.Adam(
    list(cond_encoder.parameters()) + list(unet.parameters()),
    lr=2e-4
)


Part 3-3. train_step（eps 予測）

In [ ]:
def train_step(x_past, x_future):
    x_past = x_past.to(device)
    x_future = x_future.to(device)

    B = x_future.shape[0]
    t = torch.randint(0, ddpm.T, (B,), device=device)

    x_noisy, eps_true = ddpm.q_sample(x_future, t)
    cond = cond_encoder(x_past)

    eps_pred = unet(x_noisy, t, cond)
    loss = F.mse_loss(eps_pred, eps_true)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    update_ema(unet, ema_unet)

    return loss.item()


Part 3-4. 学習ループ

In [ ]:
%%time

EPOCHS = 20

for ep in range(EPOCHS):
    for x_p, x_f in train_loader:
        loss = train_step(x_p, x_f)
    print(f"Epoch {ep+1}/{EPOCHS}  loss={loss:.4f}")


Epoch 1/20  loss=0.2989
Epoch 2/20  loss=0.2683
Epoch 3/20  loss=0.2245
Epoch 4/20  loss=0.2684
Epoch 5/20  loss=0.2391
Epoch 6/20  loss=0.2406
Epoch 7/20  loss=0.2596
Epoch 8/20  loss=0.2696
Epoch 9/20  loss=0.2353
Epoch 10/20  loss=0.2354
Epoch 11/20  loss=0.3051
Epoch 12/20  loss=0.1845
Epoch 13/20  loss=0.2694
Epoch 14/20  loss=0.2145
Epoch 15/20  loss=0.2704
Epoch 16/20  loss=0.2548
Epoch 17/20  loss=0.2845
Epoch 18/20  loss=0.2619
Epoch 19/20  loss=0.2755
Epoch 20/20  loss=0.1973
CPU times: user 1min 52s, sys: 410 ms, total: 1min 53s
Wall time: 1min 52s


Part 4. 逆拡散（DDIM, η=0）

In [ ]:
@torch.no_grad()
def reverse_diffusion(x_past):
    x_past = x_past.to(device)
    B = x_past.shape[0]
    cond = cond_encoder(x_past)

    x = torch.zeros((B, FUTURE, 2), device=device)

    for t in reversed(range(ddpm.T)):
        t_vec = torch.full((B,), t, device=device)
        eps = ema_unet(x, t_vec, cond)

        a_bar_t    = ddpm.a_bar[t_vec].view(-1,1,1)
        a_bar_prev = ddpm.a_bar_prev[t_vec].view(-1,1,1)

        x0 = (x - torch.sqrt(1-a_bar_t)*eps) / torch.sqrt(a_bar_t+1e-8)
        x0 = torch.clamp(x0, -4, 4)

        x = torch.sqrt(a_bar_prev)*x0

    return x.cpu()
